# Hiring Pipeline Optimization — Interactive Exploration

Run each cell top to bottom. By the end you'll see:
- Historical data overview: hire rate, outcome distribution, stage funnel
- How precision, recall, and cost evolve as the algorithm learns
- How each stage's threshold converges over time
- What difference `cost_weight` makes

In [ ]:
# Colab setup — clones the repo if it doesn't exist locally
import os
if not os.path.isdir('src') and not os.path.isdir('../src'):
    os.system('git clone https://github.com/fatisati/hiring-bandit.git')
    os.chdir('hiring-bandit')

In [ ]:
import sys
import os

# works from repo root (Colab) or notebooks/ subfolder (local)
src_dir = 'src' if os.path.isdir('src') else os.path.join('..', 'src')
sys.path.insert(0, src_dir)

import matplotlib.pyplot as plt
import pandas as pd

from generate_data import generate_dataset
from bandit import HiringPipeline
from evaluate import Evaluator

STAGE_COSTS = {'s0': 1, 's1': 3, 's2': 8, 's3': 15}

## 1. Generate Data

200 candidates. First 50 are historical (warm start). Remaining 150 are online.

In [ ]:
data      = generate_dataset(n=200, seed=42)
historical = [c for c in data if c['phase'] == 'historical']
online     = [c for c in data if c['phase'] == 'online']

df_hist = pd.DataFrame(historical)

hired      = df_hist['hired'].sum()
not_hired  = len(df_hist) - hired
truly_good = df_hist['ground_truth_hire'].sum()

print(f"Historical candidates : {len(df_hist)}")
print(f"Truly good (top N)    : {truly_good} ({100*truly_good//len(df_hist)}%)")
print(f"Hired                 : {hired} ({100*hired//len(df_hist)}%)")

df_hist[['candidate_id', 'true_quality', 's0_score', 's1_score', 's2_score', 's3_score',
         'ground_truth_hire', 'hired', 'outcome']].head(8)

In [ ]:
stages = ['s0', 's1', 's2', 's3']
fig, axes = plt.subplots(1, len(stages), figsize=(16, 4))
fig.suptitle('Score Distribution per Stage — green = truly good, red = not', fontsize=13)

for ax, stage in zip(axes, stages):
    scores = df_hist[[f'{stage}_score', 'ground_truth_hire']].dropna()
    good = scores[scores['ground_truth_hire'] == 1][f'{stage}_score']
    bad  = scores[scores['ground_truth_hire'] == 0][f'{stage}_score']
    ax.hist(bad,  bins=15, color='#d9534f', alpha=0.6, label='Not truly good')
    ax.hist(good, bins=15, color='#5cb85c', alpha=0.8, label='Truly good')
    ax.set_title(f'{stage}  (n={len(scores)})')
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## Naive Policy Baseline

Before the algorithm learns anything, we can measure how well the **fixed thresholds** (the company's old policy) performed on historical data. This is the baseline the online algorithm should improve upon.

In [ ]:
h_hired      = df_hist['hired'].sum()
h_truly_good = df_hist['ground_truth_hire'].sum()
h_correct    = ((df_hist['hired'] == 1) & (df_hist['ground_truth_hire'] == 1)).sum()
h_cost       = df_hist['total_cost'].sum()

h_precision    = h_correct / h_hired      if h_hired      else 0
h_recall       = h_correct / h_truly_good if h_truly_good else 0
h_cost_per_hire = h_cost   / h_hired      if h_hired      else 0

print("Naive policy (fixed thresholds) on historical data")
print("---------------------------------------------------")
print(f"Truly good candidates : {h_truly_good} / {len(df_hist)}")
print(f"Hired                 : {h_hired}")
print(f"Correct hires (TP)    : {h_correct}")
print(f"Precision             : {h_precision:.2f}")
print(f"Recall                : {h_recall:.2f}")
print(f"Total cost            : {h_cost} hrs")
print(f"Cost per hire         : {h_cost_per_hire:.1f} hrs")

## 2. Run the Pipeline

Warm starts on historical data, then processes online candidates one by one.
Thresholds are recorded after every candidate so we can plot convergence.

In [ ]:
def run_pipeline(historical, online, exploration=1.0, cost_weight=0.1, batch_size=10):
    pipeline  = HiringPipeline(stage_costs=STAGE_COSTS, exploration=exploration, cost_weight=cost_weight)
    evaluator = Evaluator(batch_size=batch_size)

    pipeline.warm_start(historical)

    threshold_history = {s: [] for s in pipeline.stages}

    for candidate in online:
        result  = pipeline.process(candidate)
        reward  = pipeline.compute_reward(result, candidate['outcome'])
        pipeline.update(result['thresholds'], result['visited_stages'], reward)
        evaluator.record(candidate, result)

        for stage in pipeline.stages:
            threshold_history[stage].append(pipeline.bandits[stage].select_threshold())

    return evaluator, threshold_history


evaluator, threshold_history = run_pipeline(historical, online)
evaluator.print_report()

## 3. Metrics Over Time

Dashed lines show the naive baseline (fixed thresholds on historical data).
If the algorithm is working, the solid lines should rise above the baseline for precision and recall, and fall below for cost per hire.

In [ ]:
batches       = [m.batch         for m in evaluator.batches]
precision     = [m.precision     for m in evaluator.batches]
recall        = [m.recall        for m in evaluator.batches]
cost_per_hire = [m.cost_per_hire for m in evaluator.batches]
total_cost    = [m.total_cost    for m in evaluator.batches]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Algorithm Performance vs Naive Baseline', fontsize=14)

# Precision
axes[0].plot(batches, precision, marker='o', color='steelblue', label='Algorithm')
axes[0].axhline(h_precision, color='steelblue', linestyle='--', alpha=0.5, label=f'Baseline ({h_precision:.2f})')
axes[0].set_title('Precision')
axes[0].set_xlabel('Batch')
axes[0].set_ylim(0, 1.05)
axes[0].legend(fontsize=8)

# Recall
axes[1].plot(batches, recall, marker='o', color='darkorange', label='Algorithm')
axes[1].axhline(h_recall, color='darkorange', linestyle='--', alpha=0.5, label=f'Baseline ({h_recall:.2f})')
axes[1].set_title('Recall')
axes[1].set_xlabel('Batch')
axes[1].set_ylim(0, 1.05)
axes[1].legend(fontsize=8)

# Cost per hire
axes[2].plot(batches, cost_per_hire, marker='o', color='crimson', label='Algorithm')
axes[2].axhline(h_cost_per_hire, color='crimson', linestyle='--', alpha=0.5, label=f'Baseline ({h_cost_per_hire:.1f})')
axes[2].set_title('Cost per Hire (hours)')
axes[2].set_xlabel('Batch')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Verdict: compare algorithm (avg of last 3 batches) vs naive baseline
last_n = min(3, len(evaluator.batches))
avg_precision    = sum(precision[-last_n:])    / last_n
avg_recall       = sum(recall[-last_n:])       / last_n
avg_cost_per_hire = sum(cost_per_hire[-last_n:]) / last_n

def verdict(algo, base, higher_is_better=True):
    diff = algo - base
    pct  = abs(diff) / base * 100 if base else 0
    improved = diff > 0 if higher_is_better else diff < 0
    symbol = '✓' if improved else '✗'
    direction = '+' if diff > 0 else ''
    return f"{symbol}  {algo:.2f}  (baseline {base:.2f}, {direction}{diff:.2f}, {pct:.0f}% {'better' if improved else 'worse'})"

print("Algorithm vs Naive Baseline  (avg of last 3 batches)")
print("=" * 55)
print(f"Precision     : {verdict(avg_precision,     h_precision,    higher_is_better=True)}")
print(f"Recall        : {verdict(avg_recall,        h_recall,       higher_is_better=True)}")
print(f"Cost per hire : {verdict(avg_cost_per_hire, h_cost_per_hire, higher_is_better=False)}")
print()

n_improved = sum([
    avg_precision    > h_precision,
    avg_recall       > h_recall,
    avg_cost_per_hire < h_cost_per_hire,
])
print(f"Overall: {n_improved}/3 metrics improved over baseline.")

## 4. Threshold Convergence

Each line shows the threshold the algorithm is currently using for that stage.
You should see it stabilise after enough candidates.

In [ ]:
stages = list(STAGE_COSTS.keys())
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Threshold Convergence per Stage', fontsize=14)

for i, stage in enumerate(stages):
    ax = axes[i // 2, i % 2]
    ax.plot(threshold_history[stage], color='steelblue', alpha=0.8)
    ax.set_title(f'Stage {stage}')
    ax.set_xlabel('Online candidate')
    ax.set_ylabel('Threshold')
    ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

## 5. Cost Weight Comparison

`cost_weight=0` — cost blind, only cares about hire quality  
`cost_weight=0.1` — penalises hours spent, pushes the algorithm to reject earlier

In [ ]:
ev_no_cost, _ = run_pipeline(historical, online, cost_weight=0.0)
ev_cost, _    = run_pipeline(historical, online, cost_weight=0.1)

b = [m.batch for m in ev_no_cost.batches]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('cost_weight=0.0 vs cost_weight=0.1', fontsize=13)

for ax, metric, label, color_pair in zip(
    axes,
    ['precision', 'recall', 'cost_per_hire'],
    ['Precision', 'Recall', 'Cost per Hire (hrs)'],
    [('steelblue', 'navy'), ('darkorange', 'saddlebrown'), ('crimson', 'darkred')]
):
    vals_0  = [getattr(m, metric) for m in ev_no_cost.batches]
    vals_01 = [getattr(m, metric) for m in ev_cost.batches]
    ax.plot(b, vals_0,  marker='o', label='cost_weight=0.0', color=color_pair[0])
    ax.plot(b, vals_01, marker='s', label='cost_weight=0.1', color=color_pair[1], linestyle='--')
    ax.set_title(label)
    ax.set_xlabel('Batch')
    ax.legend(fontsize=8)
    if metric in ('precision', 'recall'):
        ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    'metric': ['avg precision', 'avg recall', 'avg cost/hire', 'total cost'],
    'cost_weight=0.0': [
        f"{sum(m.precision     for m in ev_no_cost.batches) / len(ev_no_cost.batches):.2f}",
        f"{sum(m.recall        for m in ev_no_cost.batches) / len(ev_no_cost.batches):.2f}",
        f"{sum(m.cost_per_hire for m in ev_no_cost.batches) / len(ev_no_cost.batches):.1f} hrs",
        f"{sum(m.total_cost    for m in ev_no_cost.batches):.0f} hrs",
    ],
    'cost_weight=0.1': [
        f"{sum(m.precision     for m in ev_cost.batches) / len(ev_cost.batches):.2f}",
        f"{sum(m.recall        for m in ev_cost.batches) / len(ev_cost.batches):.2f}",
        f"{sum(m.cost_per_hire for m in ev_cost.batches) / len(ev_cost.batches):.1f} hrs",
        f"{sum(m.total_cost    for m in ev_cost.batches):.0f} hrs",
    ],
})
summary